# 5.1.1 Model 1: Linear Regression (Baseline)

Baseline model per assignment spec requirement. Trained on `X_train_scaled.csv` (log1p-transformed numeric features, per Section 3.11.2) since Linear Regression is scale-sensitive.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import sys, os
sys.path.append(os.path.dirname(os.path.abspath('__file__')))
from model_utils import evaluate_model, cross_validate_model

MODELLING_DIR = os.path.join("..", "data", "modelling")

X_train = pd.read_csv(os.path.join(MODELLING_DIR, "X_train_scaled.csv"))
X_test = pd.read_csv(os.path.join(MODELLING_DIR, "X_test_scaled.csv"))
y_train = pd.read_csv(os.path.join(MODELLING_DIR, "y_train.csv"))["price"]
y_test = pd.read_csv(os.path.join(MODELLING_DIR, "y_test.csv"))["price"]

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")


X_train: (3004, 51) | X_test: (751, 51)


## Train

In [2]:
model = LinearRegression()
model.fit(X_train, y_train)
print("Model trained.")


Model trained.


## Evaluate
Metrics computed on both train and test sets — the gap between them is needed for Section 6.2's overfitting/underfitting analysis.

In [3]:
train_metrics = evaluate_model(model, X_train, y_train, label="Train")
print()
test_metrics = evaluate_model(model, X_test, y_test, label="Test")


Train RMSE:  RM 168,244  (48.1% of median price)
Train MAE:   RM 86,459
Train MAPE:  21.2%
Train R2:    0.7366
Train MSE:   28,306,074,354

Test RMSE:  RM 194,976  (54.2% of median price)
Test MAE:   RM 92,466
Test MAPE:  19.8%
Test R2:    0.6546
Test MSE:   38,015,769,691


## Sanity check: coefficient directions vs EDA (Section 4.5.2)
Coefficient signs should broadly agree with the EDA correlation directions (e.g. Property Size positive, Property Age negative) as a basic pipeline check.

In [4]:
coef_table = pd.Series(model.coef_, index=X_train.columns).sort_values(key=abs, ascending=False)
print("Top 15 coefficients by magnitude:")
print(coef_table.head(15))


Top 15 coefficients by magnitude:
Property Size                     0.842097
State_Perak                      -0.479749
Bedroom                          -0.433820
State_Negeri_Sembilan            -0.403846
PropertyType_Service_Residence    0.304372
Bathroom                          0.289590
State_Penang                      0.287019
State_Sabah                       0.283758
State_Melaka                     -0.272305
State_Pahang                      0.245393
PropertyType_Flat                -0.237358
State_Unknown                     0.217956
Parking Lot                       0.199368
State_Sarawak                     0.197711
State_Other                       0.172923
dtype: float64


## 5-fold Cross-Validation
Run on X_train only (X_test stays untouched) to get a more robust performance estimate than a single train/test split, per Section 5.2's cross-validation requirement.

In [5]:
cv_results = cross_validate_model(LinearRegression(), X_train, y_train, n_splits=5)


5-fold CV (mean +/- std):
  RMSE:  RM 171,145 +/- 27,001  (48.7% of median price)
  MAE:   RM 88,629 +/- 5,166
  MAPE:  21.8% +/- 1.1%
  R2:    0.7215 +/- 0.0488
  MSE:   30,019,514,632 +/- 10,263,046,518
